In [ ]:
import jax
import pandas as pd

jax.config.update("jax_platform_name", "cpu")

from gould_2026.estimator import Pipeline, ArrayWithTime, CenteringEstimator
from gould_2026.dimension_reduction.prosvd import proSVD
from gould_2026.prediction.kalman_filter import StreamingKalmanFilter
from gould_2026.stim_regressor import StimRegressor, StimAutoReg
from gould_2026.datasets import Naumann24uDataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pandas as pd
import seaborn as sns
from scipy.stats import wilcoxon

plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'


In [ ]:
output1 = None
output2 = None
output3 = None
output4 = None
output5 = None

In [ ]:
rng = np.random.default_rng()
d = Naumann24uDataset(1)

target_neurons = np.unique(d.opto_stimulations.target_neuron)

d.neural_data[np.isnan(d.neural_data)] = 0

def target_neuron_to_vector(tn):
    v = target_neurons * 0
    v[target_neurons == tn] = 1
    return v

d.opto_stimulations['stim_vector'] = d.opto_stimulations['target_neuron'].apply(target_neuron_to_vector)




In [ ]:
p = Pipeline([
    CenteringEstimator(init_size=100, nan_when_uninitialized=True),
    proSVD(k=10),
])


latents = p.offline_run_on(d.neural_data, show_tqdm=True)
stim = ArrayWithTime(np.squeeze([x for x  in d.opto_stimulations['stim_vector']]), d.opto_stimulations['time'])


In [ ]:
%matplotlib inline

fig, axs = plt.subplots(nrows=2, figsize=(10,6), layout='constrained', sharex=True)
axs[0].plot(latents.t, latents, '.-')


kf = StreamingKalmanFilter(log_level=2, check_dt=True)
kf.offline_run_on(latents)
errors = ArrayWithTime.from_list(kf.log['pred_error'], squeeze_type='to_2d')
errors  = ArrayWithTime(np.linalg.norm(errors, axis=1), errors.t)
axs[1].plot(errors.t, errors, '.-')

stim_ts = []
for t1, t2 in zip(stim.t, stim.t[1:]):
    s = errors.slice_by_time(slice(t1, t2))
    stim_t = s.t[np.argmax(s)]
    stim_ts.append(max(stim_t - 2 * latents.dt, t1))

for t in stim.t:
    axs[0].axvline(t, color='red', alpha=0.3)
    axs[1].axvline(t, color='red', alpha=0.3)

for t in stim_ts:
    # axs[0].axvline(t, color='blue', alpha=0.3)
    axs[1].axvline(t, color='blue', alpha=0.3)

stim_shifted = ArrayWithTime(stim[:-1], np.array(stim_ts))

In [ ]:
srs = {
    'blind': StimRegressor(log_level=2, heed_stimuli=False, attempt_correction=False),
    'reg': StimRegressor(log_level=2, stim_delay=1*latents.dt),
}
srs['reg'].stim_autoreg = StimAutoReg(n_steps_to_consider=4)

for sr in srs.values():
    sr.offline_run_on([(stim_shifted, 'stim'), (latents, 'X')], show_tqdm=True)

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(constrained_layout=True, figsize=(10,4))

stim_df = pd.DataFrame()
for k, sr in srs.items():
    error = ArrayWithTime.from_list(sr.log['pred_error'], squeeze_type='to_2d')
    norm_error = ArrayWithTime(np.linalg.norm(error, axis=1),error.t)
    ax.plot(norm_error.t, norm_error, '.-', label=f'{k} mse_whole={np.nanmean(norm_error ** 2):.2f} mse[600:]={np.nanmean(norm_error.slice_by_time(slice(600,None))** 2):.2f}')

    pre_df = []
    for s in stim_shifted:
        i = norm_error.time_to_sample(s.t)
        e = norm_error.slice(slice(i, i+6))

        pre_df.append({
            'norm_error':e,
            'stim':s,
            't':s.t,
            'group': k,
            'stim_i': i
        })
    stim_df = pd.concat([stim_df, pd.DataFrame(pre_df)], ignore_index=True)

stim_df['error'] = stim_df['norm_error'].apply(lambda x: np.sqrt(np.mean(x**2)))
stim_df['target'] = stim_df['stim'].apply(np.argmax).astype('category')
ax.legend()
# ax.set_xlim([700, 1300])
# ax.set_ylim([0,12])

ax.set_xlabel('Time (s)')
ax.set_ylabel('norm error')

for t in stim_shifted.t:
    ax.axvline(t, color='red', alpha=0.3)

if output1 is not None:
    fig.savefig(output1)


In [ ]:
sns.stripplot(stim_df, x='group', y='error')

pivot = stim_df.pivot(index='stim_i', columns='group', values='error').dropna()
print(f"{pivot['blind'].median() = }")
print(f"{pivot['reg'].median() = }")
wilcoxon(pivot['blind'], pivot['reg'])

In [ ]:
(pivot['blind'] - pivot['reg']).median()

In [ ]:
sns.lineplot(stim_df, x='group', y='error', hue='t')

In [ ]:

sns.scatterplot(stim_df, x='t', y='error', hue='group')

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(5,5))
pivot = stim_df.pivot(index='stim_i', columns='group', values='error')
sns.scatterplot(pivot, x='blind', y='reg', ax=ax)
ax.axis('equal')

lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1]),
]
ax.plot(lims, lims, '--', color='gray', zorder=0)

In [ ]:
kf_error = ArrayWithTime.from_list(srs['reg'].log['pred_error'])

fig, ax = plt.subplots(constrained_layout=True, figsize=(8,8))

preq_errors = ArrayWithTime.from_list(srs['reg'].stim_reg.log['preq_errors'], squeeze_type='to_2d')
preq_errors = np.linalg.norm(preq_errors,axis=1)
n_observed = srs['reg'].stim_reg.n_observed


ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=1)
error_scatter = ax.scatter(srs['reg'].stim_reg.input_histories[0][:n_observed,0], srs['reg'].stim_reg.input_histories[0][:n_observed,1], c=preq_errors, cmap='plasma')
ax.set_xlim(right=20)
fig.colorbar(error_scatter)


if output2 is not None:
    fig.savefig(output2)



In [ ]:
%matplotlib inline
fig, axs = plt.subplots(constrained_layout=True, figsize=(10,5), ncols=2)

n = 25
l = 1
r = 7
time_slice = slice(stim_shifted.t[n] -l, stim_shifted.t[n] + r)

ax = axs[1]
ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=1)
# ax.plot(latents[:,0], latents[:,1], '-')
ax.set_xlim(right=20)
idx = np.argmin(np.linalg.norm(latents - srs['reg'].stim_reg.input_histories[0][0],axis=1))
assert np.linalg.norm(latents.slice(idx) - srs['reg'].stim_reg.input_histories[0][0]) < 1e-10
idx = latents.time_to_sample(srs['reg'].stim_reg.input_histories[2][:n_observed, 0]) -1
latents_at_stims = latents.slice(idx).slice_by_time(time_slice)
ax.scatter(latents_at_stims[:,0], latents_at_stims[:,1], c='r')
latent_slice = latents.slice_by_time(time_slice)
ax.plot(latent_slice[:,0], latent_slice[:,1])

ax = axs[0]
# full_slice = d.neural_data.slice_by_time(time_slice)
# ax.plot(full_slice.t, full_slice, 'k');
ax.plot(latent_slice.t, latent_slice, 'k');
for t in latents_at_stims.t:
    ax.axvline(t, color='r', alpha=0.5)
ax.set_xlabel('Time (s)')


if output3 is not None:
    fig.savefig(output3)



In [ ]:
plt.plot(stim_shifted.t + latents.dt - srs['reg'].stim_reg.input_histories[2][:n_observed, 0])
# TODO: resolve this!!

## Behavior analysis

In [ ]:
d = Naumann24uDataset(1)

target_neurons = np.unique(d.opto_stimulations.target_neuron)

d.neural_data[np.isnan(d.neural_data)] = 0


def target_neuron_to_vector(tn):
    v = target_neurons * 0
    v[target_neurons == tn] = 1
    return v


d.opto_stimulations['stim_vector'] = d.opto_stimulations['target_neuron'].apply(target_neuron_to_vector)

p = Pipeline([
    CenteringEstimator(init_size=100, nan_when_uninitialized=True),
    proSVD(k=10),
])

latents = p.offline_run_on(d.neural_data, show_tqdm=True)
stim = ArrayWithTime(np.squeeze([x for x in d.opto_stimulations['stim_vector']]), d.opto_stimulations['time'])


In [ ]:

beh = d.behavioral_data
fig, ax = plt.subplots(constrained_layout=True, figsize=(10,5))
ax.plot(beh.t, beh)

for t in d.opto_stimulations.time:
    ax.axvline(t, color='red', alpha=0.1)


if output4 is not None:
    fig.savefig(output4)

In [ ]:
srs = {
    'blind': StimRegressor(log_level=2, heed_stimuli=False, attempt_correction=False, error_on_missed_stim=False),
    'reg': StimRegressor(log_level=2, stim_delay=1*latents.dt, error_on_missed_stim=False),
}
srs['reg'].stim_autoreg = StimAutoReg(n_steps_to_consider=4)

for sr in srs.values():
    sr.offline_run_on([(stim_shifted, 'stim'), (ArrayWithTime(np.column_stack([beh,beh]), beh.t), 'X')], show_tqdm=True)

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(constrained_layout=True, figsize=(10,4))

for k, sr in srs.items():
    args = (slice(None), slice(0,1))
    error = ArrayWithTime.from_list(sr.log['pred_error'], squeeze_type='to_2d')
    error = ArrayWithTime(error[:,0:1], error.t)
    norm_error = ArrayWithTime(np.linalg.norm(error, axis=1),error.t)
    ax.plot(error.t, error, '.-', label=f'{k} mse_whole={np.nanmean(error ** 2):.2f} mse[600:]={np.nanmean(error.slice_by_time(slice(600,None))** 2):.6f}')

ax.legend()
# ax.set_xlim([700, 1300])
# ax.set_ylim([0,12])

ax.set_xlabel('Time (s)')
ax.set_ylabel('norm error')

for t in stim_shifted.t:
    ax.axvline(t, color='red', alpha=0.3)

if output1 is not None:
    fig.savefig(output1)


In [ ]:
kf_error = ArrayWithTime.from_list(srs['reg'].log['pred_error'])
kf_error = ArrayWithTime(kf_error[:,0,0:1], kf_error.t)

behavior_per_stim = []
for a,b in zip(d.opto_stimulations.time, d.opto_stimulations.time[1:]):
    s = slice(a,b)
    single_stim_behavior = beh.slice_by_time(s)
    idx = np.nanargmax(np.abs(single_stim_behavior))
    behavior_per_stim.append(ArrayWithTime(single_stim_behavior[idx], single_stim_behavior.t[idx]))
behavior_per_stim = ArrayWithTime.from_list(behavior_per_stim)

behavior_per_stim = behavior_per_stim.slice(np.abs(behavior_per_stim[:,0]) > 0.05)

latent_slice = latents.slice_by_time(behavior_per_stim.t)

fig, ax = plt.subplots(constrained_layout=True, figsize=(8,8))
ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=1)
error_scatter = ax.scatter(latent_slice[:,0], latent_slice[:,1], c=behavior_per_stim, cmap='plasma')
ax.set_xlim(right=20)
fig.colorbar(error_scatter)



In [ ]:

def f(point):
    idx = np.argmin(np.linalg.norm(latent_slice[:,0:2] - point, axis=1))
    return behavior_per_stim[idx]


def f2(point, bottom=-1.24, top=2.4):
    point = point /8
    quadratic = point[0]**2 - 1.8*point[1]**2
    surface = np.tanh(quadratic) * (top-bottom)/2
    surface = surface - (-(top-bottom)/2 - bottom)
    return  surface




# Get axis limits from the scatterplot cell above
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Build a grid over the same domain
n_grid = 200
x_grid = np.linspace(xlim[0], xlim[1], n_grid)
y_grid = np.linspace(ylim[0], ylim[1], n_grid)
xx, yy = np.meshgrid(x_grid, y_grid)
grid_points = np.vstack([xx.ravel(), yy.ravel()])

# Evaluate function (KDE of latent points) at grid points
zz_data = []
zz_functional = []
for point in tqdm(grid_points.T):
    zz_data.append(f(point))
    zz_functional.append(f2(point))
zz_data = np.array(zz_data).reshape((n_grid, n_grid))
zz_functional = np.array(zz_functional).reshape((n_grid, n_grid))



vmin = min(zz_data.flatten().min(), zz_functional.flatten().min())
vmax = min(zz_data.flatten().max(), zz_functional.flatten().max())

fig, axs = plt.subplots(ncols=2, constrained_layout=True, figsize=(10, 5))
ax_hm = axs[0]
mesh = ax_hm.pcolormesh(xx, yy, zz_data, cmap='plasma', shading='auto', vmin=vmin, vmax=vmax)
ax_hm.scatter(latents[:, 0], latents[:, 1], c='gray', alpha=0.5, s=1)
ax_hm.set_xlim(xlim)
ax_hm.set_ylim(ylim)
# fig.colorbar(mesh, ax=ax_hm)

ax_functional = axs[1]
mesh = ax_functional.pcolormesh(xx, yy, zz_functional, cmap='plasma', shading='auto', vmin=vmin, vmax=vmax)
ax_functional.scatter(latents[:, 0], latents[:, 1], c='gray', alpha=0.5, s=1)
ax_functional.set_xlim(xlim)
ax_functional.set_ylim(ylim);



## Discretized behavior analysis

In [ ]:

b2 = []
for a,b in zip(d.opto_stimulations.time, d.opto_stimulations.time[1:]):
    s = slice(a,b)
    b = beh.slice_by_time(s)
    b2.append(ArrayWithTime([b.max() > .5, b.min() < -.5], a + .1))

b2 = ArrayWithTime.from_list(b2, squeeze_type='to_2d')

fig, axs = plt.subplots(constrained_layout=True, nrows=2, figsize=(10,8), height_ratios=[1,10])
axs[0].matshow(b2.T)
axs[1].matshow(stim.T)


In [ ]:
%matplotlib inline

sr1 = StimRegressor(log_level=2, autoreg=StreamingKalmanFilter(steps_between_refits=2), heed_stimuli=False, attempt_correction=False)
sr2 = StimRegressor(log_level=2, autoreg=StreamingKalmanFilter(steps_between_refits=2))

sr1.offline_run_on([(ArrayWithTime.from_notime(stim).slice(slice(4,None)), 'stim'), (ArrayWithTime.from_notime(b2), 'X')], show_tqdm=True)
sr2.offline_run_on([(ArrayWithTime.from_notime(stim).slice(slice(4,None)), 'stim'), (ArrayWithTime.from_notime(b2), 'X')], show_tqdm=True)

error = ArrayWithTime.from_list(sr1.log['pred_error'], squeeze_type='to_2d')
ne = np.linalg.norm(error, axis=1)
print(me1 :=np.nanmean(ne[40:]**2))
plt.plot(error.t, ne)

error = ArrayWithTime.from_list(sr2.log['pred_error'], squeeze_type='to_2d')
ne = np.linalg.norm(error, axis=1)
print(me2:=np.nanmean(ne[40:]**2))
plt.plot(error.t, ne)


plt.legend([f'blind, mean(error[40:]**2)={me1:.4f}', f'reg, mean(error[40:]**2) = {me2:.4f}'])



In [ ]:
ne = ArrayWithTime(ne, stim.slice(slice(3,None)).t)

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(8,8))

# preq_errors = ArrayWithTime.from_list(srs['reg'].stim_reg.log['preq_errors'], squeeze_type='to_2d')
# preq_errors = np.linalg.norm(preq_errors,axis=1)
n_observed = srs['reg'].stim_reg.n_observed


ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=1)

ls = np.array(latents.slice_by_time(ne.t))
error_scatter = ax.scatter(ls[:,0], ls[:,1], c=np.array(ne), cmap='plasma')
ax.set_xlim(right=20)
fig.colorbar(error_scatter)

if output5 is not None:
    fig.savefig(output5)
